# Data Cleaning — BC_A&A_with_ATD.csv

This notebook cleans the raw dataset and saves a clean version to `data/processed/clean.parquet`.

## Steps
| # | Step | What it does |
|---|------|--------------|
| 1 | Load raw data | Parse `\\N` as NaN, set dtypes, parse timestamps |
| 2 | Inspect raw missingness | Count NaN per column before any cleaning |
| 3 | ATD outlier analysis | IQR fence, z-score, p99 cap strategy |
| 4 | Remove impossible rows | Negative ATD, timestamp inversions |
| 5 | Impute distances | Territory × courier_flow × hour-block median |
| 6 | Impute driver_uuid | Flag-only: mark as `UNKNOWN` |
| 7 | Cap ATD outliers | Winsorise at p99 per territory |
| 8 | Drop residual nulls | Drop rows still missing ATD or timestamps |
| 9 | Save clean dataset | Write `data/processed/clean.parquet` |

---
## 0 · Setup & Load

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (14, 4)})

RAW_PATH   = Path('../data/raw/BC_A&A_with_ATD.csv')
OUT_PATH   = Path('../data/processed/clean.parquet')
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
dtype_map = {
    'region': 'category',
    'territory': 'category',
    'country_name': 'category',
    'courier_flow': 'category',
    'geo_archetype': 'category',
    'merchant_surface': 'category',
}
parse_cols = [
    'restaurant_offered_timestamp_utc',
    'order_final_state_timestamp_local',
    'eater_request_timestamp_local',
]

# \N is MySQL's NULL export representation — treat it as NaN on load
df = pd.read_csv(
    RAW_PATH,
    dtype=dtype_map,
    parse_dates=parse_cols,
    na_values=['\\N', 'NULL', 'None', ''],
)

print(f'Rows    : {len(df):,}')
print(f'Columns : {df.shape[1]}')
print(f'Memory  : {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB')
df.dtypes.to_frame('dtype')

---
## 1 · Raw Missingness Snapshot

Baseline before any cleaning — so we can compare after each step.

In [ ]:
def missing_report(frame, label=''):
    m = (
        frame.isnull().sum()
        .rename('n_missing')
        .to_frame()
        .assign(pct=lambda x: (x['n_missing'] / len(frame) * 100).round(4))
        .sort_values('n_missing', ascending=False)
    )
    if label:
        print(f'--- {label} ({len(frame):,} rows) ---')
    return m

raw_missing = missing_report(df, 'RAW')
display(raw_missing[raw_missing['n_missing'] > 0])

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
cols_with_na = raw_missing[raw_missing['n_missing'] > 0]
cols_with_na['pct'].plot(kind='bar', ax=ax, color='coral', edgecolor='none')
ax.set_title('Raw Missing Value Rate by Column (%)', fontweight='bold')
ax.set_ylabel('% Missing')
ax.tick_params(axis='x', rotation=35)
for i, v in enumerate(cols_with_na['pct']):
    ax.text(i, v + 0.05, f'{v:.2f}%', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Verify \N was fully parsed — check if any string '\N' survived
str_cols = df.select_dtypes(include='object').columns.tolist()
for col in str_cols:
    leftover = (df[col] == '\\N').sum()
    if leftover > 0:
        print(f'  WARNING: {col} still has {leftover:,} literal \\N strings — replacing now')
        df[col] = df[col].replace('\\N', np.nan)
    else:
        print(f'  OK: {col} — no residual \\N strings')

print('\n\\N scan complete.')

---
## 2 · ATD Outlier Analysis

Three complementary approaches:
- **IQR fence** — classic Tukey 1.5× and 3× (mild / extreme)
- **Z-score** — how many std devs from the mean
- **Percentile view** — p95, p99, p99.9 thresholds

We will **not drop** outliers at this stage — we cap them later (Step 5) to preserve row count.

In [ ]:
atd = df['ATD'].dropna()

print('ATD descriptive statistics:')
print(atd.describe(percentiles=[.01, .05, .1, .25, .5, .75, .9, .95, .99, .999]).round(2))

In [ ]:
Q1  = atd.quantile(0.25)
Q3  = atd.quantile(0.75)
IQR = Q3 - Q1

mild_lo,  mild_hi  = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
extr_lo,  extr_hi  = Q1 - 3.0 * IQR, Q3 + 3.0 * IQR

print(f'IQR = {IQR:.2f} min  (Q1={Q1:.2f}, Q3={Q3:.2f})')
print()
print(f'Mild  fence : [{mild_lo:.2f}, {mild_hi:.2f}] min')
print(f'  Rows below lower : {(atd < mild_lo).sum():,} ({(atd < mild_lo).mean()*100:.3f}%)')
print(f'  Rows above upper : {(atd > mild_hi).sum():,} ({(atd > mild_hi).mean()*100:.3f}%)')
print()
print(f'Extreme fence: [{extr_lo:.2f}, {extr_hi:.2f}] min')
print(f'  Rows below lower : {(atd < extr_lo).sum():,} ({(atd < extr_lo).mean()*100:.3f}%)')
print(f'  Rows above upper : {(atd > extr_hi).sum():,} ({(atd > extr_hi).mean()*100:.3f}%)')

In [ ]:
z = np.abs(stats.zscore(atd))

print('Z-score outlier counts:')
for threshold in [2, 3, 4, 5]:
    n = (z > threshold).sum()
    print(f'  |z| > {threshold}: {n:,} rows ({n/len(atd)*100:.4f}%)')

In [ ]:
thresholds = [60, 90, 120, 180, 240, 360, 480, 1440]
pct_data = []
for t in thresholds:
    n = (atd > t).sum()
    pct_data.append({'ATD_threshold_min': t, 'rows_above': n, 'pct_above': n/len(atd)*100})

print('Rows exceeding common ATD thresholds:')
display(pd.DataFrame(pct_data).round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 4))

# Full distribution (log y)
axes[0].hist(atd.clip(0, 500), bins=100, color='steelblue', edgecolor='none', log=True)
axes[0].axvline(mild_hi, color='orange', linewidth=1.5, label=f'IQR 1.5× ({mild_hi:.0f} min)')
axes[0].axvline(extr_hi, color='red',    linewidth=1.5, label=f'IQR 3.0× ({extr_hi:.0f} min)')
axes[0].set_title('ATD Distribution (log y, clipped 500 min)')
axes[0].set_xlabel('ATD (min)')
axes[0].legend(fontsize=8)

# Box plot
axes[1].boxplot(atd, vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6),
                flierprops=dict(marker='.', markersize=1, alpha=0.3, color='tomato'))
axes[1].set_title('ATD Boxplot (all values)')
axes[1].set_ylabel('ATD (min)')
axes[1].set_xticks([])

# Zoom into tail (> p95)
p95 = atd.quantile(0.95)
tail = atd[atd > p95]
axes[2].hist(tail, bins=60, color='tomato', edgecolor='none')
axes[2].axvline(atd.quantile(0.99),  color='black', linestyle='--', linewidth=1.5, label='p99')
axes[2].axvline(atd.quantile(0.999), color='purple', linestyle='--', linewidth=1.5, label='p99.9')
axes[2].set_title(f'ATD Tail (> p95 = {p95:.0f} min)')
axes[2].set_xlabel('ATD (min)')
axes[2].legend(fontsize=8)

plt.suptitle('ATD Outlier Analysis', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Outlier rate by territory — helps decide whether to cap globally or per-territory
atd_territory = (
    df[df['ATD'].notna()]
    .groupby('territory', observed=True)['ATD']
    .agg(
        count='count',
        median='median',
        p95=lambda x: x.quantile(0.95),
        p99=lambda x: x.quantile(0.99),
        pct_over_120=lambda x: (x > 120).mean() * 100,
        pct_over_240=lambda x: (x > 240).mean() * 100,
    )
    .sort_values('pct_over_120', ascending=False)
    .round(2)
)
print('ATD outlier profile by territory:')
display(atd_territory)

---
## 3 · Remove Impossible Rows

Rows that are logically invalid and cannot be corrected:
- **ATD ≤ 0** — delivery completed before restaurant received the order
- **Timestamp inversion** — `order_final_state < restaurant_offered` (after timezone correction)
- **ATD > 1440 min (>24 h)** — almost certainly a data pipeline error, not a real trip

In [ ]:
n_start = len(df)

# Mask: rows to KEEP
keep_atd = df['ATD'].isna() | ((df['ATD'] > 0) & (df['ATD'] <= 1440))

# Timestamp inversion check (UTC-6 offset applied to local timestamps)
UTC_OFFSET = pd.Timedelta(hours=6)
has_all_ts = (
    df['restaurant_offered_timestamp_utc'].notna() &
    df['order_final_state_timestamp_local'].notna()
)
final_utc   = df['order_final_state_timestamp_local'] + UTC_OFFSET
offered_utc = df['restaurant_offered_timestamp_utc']

# Keep rows where we can't check (missing ts) OR where ordering is valid
keep_ts = ~has_all_ts | (final_utc > offered_utc)

removed_atd = (~keep_atd).sum()
removed_ts  = (~keep_ts).sum()

print(f'Rows with impossible ATD (≤0 or >1440 min) : {removed_atd:,}')
print(f'Rows with timestamp inversion               : {removed_ts:,}')

df = df[keep_atd & keep_ts].copy()
print(f'\nRows remaining: {len(df):,}  (removed {n_start - len(df):,} impossible rows)')

---
## 4 · Handle `\N` / NaN in `driver_uuid`

`driver_uuid` can be null for several legitimate reasons:
- Order cancelled before assignment
- Re-dispatch in progress
- Data pipeline gap

**Strategy**: fill with `'UNKNOWN'` so the column stays usable in groupbys and joins without dropping rows.

In [ ]:
n_missing_driver = df['driver_uuid'].isna().sum()
print(f'driver_uuid missing: {n_missing_driver:,} ({n_missing_driver/len(df)*100:.3f}%)')

if n_missing_driver > 0:
    # Add a flag column before filling
    df['driver_uuid_missing'] = df['driver_uuid'].isna().astype('int8')
    df['driver_uuid'] = df['driver_uuid'].fillna('UNKNOWN')
    print(f'  → Filled with "UNKNOWN", added binary flag column `driver_uuid_missing`')
else:
    df['driver_uuid_missing'] = 0
    print('  → No missing driver_uuid values.')

# Show ATD profile for UNKNOWN vs known drivers
if 'driver_uuid_missing' in df.columns and df['driver_uuid_missing'].sum() > 0:
    print('\nATD profile — UNKNOWN vs known drivers:')
    display(
        df.groupby('driver_uuid_missing')['ATD']
        .agg(['count', 'median', 'mean', lambda x: x.quantile(0.95)])
        .rename(columns={'<lambda_0>': 'p95'})
        .round(2)
    )

---
## 5 · Impute Missing Distances

Both `pickup_distance` and `dropoff_distance` have ~2.5% missing values.
Imputation strategy: **territory × courier_flow × hour-block median**.

Why this grouping?
- Territory captures geography (distance ranges vary city to city)
- courier_flow captures mode (bike vs car have different delivery radii)
- Hour block captures time-of-day demand patterns

Fallback chain: territory+flow+hour → territory+flow → territory → global median

In [ ]:
# Create hour_local from eater_request (already local time UTC-6)
df['hour_local'] = df['eater_request_timestamp_local'].dt.hour

# 4-hour blocks: 0=night(0-3), 1=morning(4-7), 2=midday(8-11),
#                3=afternoon(12-15), 4=evening(16-19), 5=night(20-23)
df['hour_block'] = (df['hour_local'] // 4).astype('Int8')

print('Hour block distribution:')
block_labels = {0:'00-03', 1:'04-07', 2:'08-11', 3:'12-15', 4:'16-19', 5:'20-23'}
display(
    df['hour_block'].value_counts().sort_index()
    .rename(index=block_labels)
    .rename('count')
    .to_frame()
)

In [ ]:
def impute_with_fallback(series, groups, df_ref):
    """
    Fill NaN in `series` using progressively coarser group medians.
    groups: list of column lists, from finest to coarsest.
    """
    result = series.copy()
    col_name = series.name

    for group_cols in groups:
        still_na = result.isna()
        if still_na.sum() == 0:
            break
        medians = df_ref.groupby(group_cols, observed=True)[col_name].median()
        fill_values = df_ref[group_cols].apply(
            lambda row: medians.get(tuple(row) if len(group_cols) > 1 else row.iloc[0], np.nan),
            axis=1
        )
        result = result.fillna(fill_values)
        print(f'  After groupby {group_cols}: {result.isna().sum():,} still missing')

    # Final fallback: global median
    if result.isna().sum() > 0:
        global_med = df_ref[col_name].median()
        result = result.fillna(global_med)
        print(f'  After global median fallback: {result.isna().sum():,} still missing')

    return result


fallback_groups = [
    ['territory', 'courier_flow', 'hour_block'],
    ['territory', 'courier_flow'],
    ['territory'],
]

for dist_col in ['pickup_distance', 'dropoff_distance']:
    n_before = df[dist_col].isna().sum()
    print(f'\n{dist_col}: {n_before:,} missing before imputation')
    df[dist_col] = impute_with_fallback(df[dist_col], fallback_groups, df)
    n_after = df[dist_col].isna().sum()
    print(f'  → {n_after:,} missing after imputation')

In [ ]:
# Compare distributions before/after imputation using the original null mask
raw_df = pd.read_csv(
    RAW_PATH,
    usecols=['pickup_distance', 'dropoff_distance'],
    na_values=['\\N', 'NULL', 'None', ''],
)
orig_na_mask = raw_df['pickup_distance'].isna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, col in zip(axes, ['pickup_distance', 'dropoff_distance']):
    # Rows that were originally non-null
    original_vals = df.loc[~orig_na_mask.reindex(df.index, fill_value=False), col]
    imputed_vals  = df.loc[ orig_na_mask.reindex(df.index, fill_value=False), col]

    cap = original_vals.quantile(0.99)
    ax.hist(original_vals.clip(0, cap), bins=60, alpha=0.6, color='steelblue', label='original', density=True)
    ax.hist(imputed_vals.clip(0, cap),  bins=60, alpha=0.6, color='tomato',    label='imputed',  density=True)
    ax.set_title(f'{col} — original vs imputed (density, ≤p99)')
    ax.set_xlabel('km')
    ax.legend()

plt.suptitle('Distance Imputation Quality Check', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6 · Cap ATD Outliers (Winsorisation)

Rather than dropping extreme ATD rows, we cap them at the **99th percentile per territory**.
This preserves row count while limiting leverage of extreme values on models.

The original `ATD` column is kept; `ATD_capped` is added for modeling use.

In [ ]:
# Per-territory p99 cap
territory_p99 = (
    df[df['ATD'].notna()]
    .groupby('territory', observed=True)['ATD']
    .quantile(0.99)
    .rename('atd_p99')
)

print('Per-territory ATD p99 caps:')
display(territory_p99.sort_values(ascending=False).round(2).to_frame())

# Map cap to each row and apply
df = df.join(territory_p99, on='territory')
df['ATD_capped'] = df['ATD'].clip(upper=df['atd_p99'])
df.drop(columns='atd_p99', inplace=True)

n_capped = (df['ATD'] > df['ATD_capped']).sum()
print(f'\nRows capped: {n_capped:,} ({n_capped/len(df)*100:.3f}%)')
print(f'ATD_capped stats:')
print(df['ATD_capped'].describe(percentiles=[.5, .75, .95, .99]).round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

cap_val = df['ATD_capped'].max()
axes[0].hist(df['ATD'].clip(0, cap_val + 20),         bins=80, alpha=0.6, color='tomato',    label='ATD original', density=True)
axes[0].hist(df['ATD_capped'].clip(0, cap_val + 20),  bins=80, alpha=0.6, color='steelblue', label='ATD capped',   density=True)
axes[0].set_title('ATD — original vs capped (density)')
axes[0].set_xlabel('min')
axes[0].legend()

# Tail zoom
tail_start = df['ATD'].quantile(0.95)
axes[1].hist(df.loc[df['ATD'] > tail_start, 'ATD'],         bins=60, alpha=0.6, color='tomato',    label='ATD original', density=True)
axes[1].hist(df.loc[df['ATD'] > tail_start, 'ATD_capped'],  bins=60, alpha=0.6, color='steelblue', label='ATD capped',   density=True)
axes[1].set_title(f'Tail zoom (ATD > p95 = {tail_start:.0f} min)')
axes[1].set_xlabel('min')
axes[1].legend()

plt.suptitle('ATD Winsorisation', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7 · Drop Residual Nulls

After imputation, any remaining nulls in critical columns are non-recoverable and must be dropped.

In [ ]:
critical_cols = [
    'ATD',
    'restaurant_offered_timestamp_utc',
    'eater_request_timestamp_local',
    'order_final_state_timestamp_local',
    'pickup_distance',
    'dropoff_distance',
]

n_before = len(df)
df.dropna(subset=critical_cols, inplace=True)
n_dropped = n_before - len(df)

print(f'Dropped {n_dropped:,} rows with nulls in critical columns.')
print(f'Rows remaining: {len(df):,}')

# Final missing report
print('\nFinal missing value report:')
display(missing_report(df, 'CLEAN')[lambda x: x['n_missing'] > 0])

---
## 8 · Final Sanity Check & Summary

In [ ]:
checks = {
    'ATD ≤ 0'                  : (df['ATD'] <= 0).sum(),
    'ATD > 1440 min'           : (df['ATD'] > 1440).sum(),
    'pickup_distance < 0'      : (df['pickup_distance'] < 0).sum(),
    'dropoff_distance < 0'     : (df['dropoff_distance'] < 0).sum(),
    'pickup_distance missing'  : df['pickup_distance'].isna().sum(),
    'dropoff_distance missing' : df['dropoff_distance'].isna().sum(),
    'ATD missing'              : df['ATD'].isna().sum(),
    'driver_uuid missing'      : (df['driver_uuid'] == 'UNKNOWN').sum(),
    'Timestamp inversions'     : (
        (df['order_final_state_timestamp_local'] + UTC_OFFSET)
        < df['restaurant_offered_timestamp_utc']
    ).sum(),
}

check_df = pd.DataFrame(list(checks.items()), columns=['Check', 'Count'])
check_df['Status'] = check_df['Count'].apply(lambda n: '✅ PASS' if n == 0 else f'⚠️  {n:,}')

print('Final Sanity Checks on Clean Dataset:')
display(check_df)

In [ ]:
raw_count = pd.read_csv(RAW_PATH, usecols=['ATD']).shape[0]

print('=' * 55)
print('  DATA CLEANING SUMMARY')
print('=' * 55)
print(f'  Raw rows         : {raw_count:>10,}')
print(f'  Clean rows       : {len(df):>10,}')
print(f'  Rows removed     : {raw_count - len(df):>10,}  ({(raw_count - len(df))/raw_count*100:.2f}%)')
print('─' * 55)
print(f'  ATD range        : {df["ATD"].min():.2f} – {df["ATD"].max():.2f} min')
print(f'  ATD_capped range : {df["ATD_capped"].min():.2f} – {df["ATD_capped"].max():.2f} min')
print(f'  Median ATD       : {df["ATD"].median():.1f} min')
print(f'  Distances imputed: {df["driver_uuid_missing"].sum():,} trips had missing driver_uuid')
print('=' * 55)

---
## 9 · Save Clean Dataset

In [ ]:
# Drop helper columns not needed downstream
cols_to_drop = ['hour_block']  # hour_local is useful, keep it
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

df.to_parquet(OUT_PATH, index=False, engine='pyarrow')

size_mb = OUT_PATH.stat().st_size / 1024**2
print(f'Saved clean dataset to: {OUT_PATH}')
print(f'  Shape  : {df.shape}')
print(f'  Size   : {size_mb:.1f} MB')
print(f'  Columns: {list(df.columns)}')